In [1]:
import json, glob, os
import numpy as np, pandas as pd

paths = {}
for name in ["results_full", "stability_results", "focal_stability_results"]:
    hits = glob.glob(f"/kaggle/**/{name}.json", recursive=True)
    paths[name] = hits[0] if hits else None
    print(f"{name:28s} -> {paths[name]}")

data = {k: json.load(open(v)) for k, v in paths.items() if v}
for k, v in data.items():
    print(f"\n{k}: {len(v)} records")
    print("  keys:", sorted(v[0].keys()))

results_full                 -> /kaggle/input/datasets/findme77/nids-stability-results/results_full.json
stability_results            -> /kaggle/input/datasets/findme77/nids-stability-results/stability_results.json
focal_stability_results      -> /kaggle/input/datasets/findme77/nids-stability-results/focal_stability_results.json

results_full: 30 records
  keys: ['accuracy', 'balanced_accuracy', 'batch1_latency_ms', 'confusion_matrix', 'final_train_loss', 'loss_curve', 'macro_f1', 'mcc', 'n_params', 'n_test', 'n_train', 'per_class', 'protocol', 'sec_per_epoch', 'seed', 'strategy', 'total_train_sec', 'wall_sec', 'weighted_f1']

stability_results: 40 records
  keys: ['accuracy', 'balanced_accuracy', 'best_val_accuracy', 'best_val_epoch', 'confusion_matrix', 'final_train_loss', 'final_val_accuracy', 'generalisation_gap', 'loss_curve', 'macro_f1', 'max_pred_share', 'mcc', 'n_empty_pred_classes', 'protocol', 'rescued_accuracy', 'rescued_balanced_accuracy', 'rescued_mcc', 'seed', 'train_accu

In [2]:
import numpy as np, pandas as pd

def summarise(records, label):
    t  = np.array([r["accuracy"] for r in records])
    s  = np.array([r["rescued_accuracy"] for r in records])
    tb = np.array([r["balanced_accuracy"] for r in records])
    sb = np.array([r["rescued_balanced_accuracy"] for r in records])
    ep = np.array([r["best_val_epoch"] for r in records])
    med = np.median(t)
    return {
        "setup": label, "n": len(t),
        "final_acc": t.mean(), "final_sd": t.std(ddof=1),
        "resc_acc": s.mean(),  "resc_sd": s.std(ddof=1),
        "gain": (s - t).mean(),
        "var_red": t.std(ddof=1) / s.std(ddof=1),
        "final_bal": tb.mean(), "resc_bal": sb.mean(),
        "ep40": int((ep == 40).sum()),
        "ep_min": int(ep.min()), "ep_max": int(ep.max()),
        "collapse": int((t < 0.85 * med).sum()),
    }

stab  = data["stability_results"]
focal = data["focal_stability_results"]
rows = [
    summarise([r for r in stab if r["protocol"] == "A"], "CE / protocol A"),
    summarise([r for r in stab if r["protocol"] == "B"], "CE / protocol B"),
    summarise(focal, "Focal / protocol B"),
]
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

             setup  n  final_acc  final_sd  resc_acc  resc_sd   gain  var_red  final_bal  resc_bal  ep40  ep_min  ep_max  collapse
   CE / protocol A 20     0.8139    0.0543    0.8759   0.0075 0.0620   7.2178     0.6301    0.6731     1      19      40         2
   CE / protocol B 20     0.8158    0.0614    0.8737   0.0091 0.0580   6.7501     0.6292    0.6718     2      19      40         1
Focal / protocol B 20     0.7653    0.0096    0.7675   0.0089 0.0022   1.0787     0.7089    0.7100    10      16      40         0


In [3]:
from scipy import stats

for lbl, recs in [("CE protocol A", [r for r in stab if r["protocol"]=="A"]),
                  ("CE protocol B", [r for r in stab if r["protocol"]=="B"]),
                  ("Focal protocol B", focal)]:
    t = np.array([r["accuracy"] for r in recs])
    s = np.array([r["rescued_accuracy"] for r in recs])
    w = stats.wilcoxon(s, t)
    print(f"{lbl:18s} improved {int((s>t).sum())}/{len(t)}  "
          f"W={w.statistic:.1f}  p={w.pvalue:.2e}  mean gain {(s-t).mean():+.4f}")

# patience-based early stopping — the non-oracle version
def early_stop(r, patience):
    va = r["val_acc_curve"]
    best, best_ep, wait = -1, 0, 0
    for i, v in enumerate(va):
        if v > best: best, best_ep, wait = v, i, 0
        else:
            wait += 1
            if wait >= patience: return best_ep + 1, i + 1
    return best_ep + 1, len(va)

print()
for pat in [5, 10]:
    for lbl, recs in [("CE-A", [r for r in stab if r["protocol"]=="A"]),
                      ("CE-B", [r for r in stab if r["protocol"]=="B"])]:
        picked = [early_stop(r, pat)[0] for r in recs]
        stopped = [early_stop(r, pat)[1] for r in recs]
        oracle  = [r["best_val_epoch"] for r in recs]
        match   = sum(p == o for p, o in zip(picked, oracle))
        print(f"patience={pat:2d} {lbl}: stops at ep {np.mean(stopped):.1f}, "
              f"picks ep {np.mean(picked):.1f}, matches oracle {match}/{len(recs)}")

CE protocol A      improved 19/20  W=0.0  p=1.32e-04  mean gain +0.0620
CE protocol B      improved 18/20  W=0.0  p=1.96e-04  mean gain +0.0580
Focal protocol B   improved 10/20  W=0.0  p=5.06e-03  mean gain +0.0022

patience= 5 CE-A: stops at ep 14.8, picks ep 9.8, matches oracle 0/20
patience= 5 CE-B: stops at ep 15.3, picks ep 10.3, matches oracle 1/20
patience=10 CE-A: stops at ep 28.2, picks ep 19.6, matches oracle 7/20
patience=10 CE-B: stops at ep 29.2, picks ep 20.4, matches oracle 10/20


In [4]:
import numpy as np, pandas as pd

CLASS_NAMES = ['Benign','BruteForce','DDoS','DoS','Mirai','Recon','Spoofing','Web']

def recalls_from_cm(cm):
    cm = np.array(cm)
    return cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

def perclass_table(recs, label):
    rec = np.array([recalls_from_cm(r["confusion_matrix"]) for r in recs])
    return pd.DataFrame({
        "class": CLASS_NAMES,
        f"{label}_mean": rec.mean(axis=0),
        f"{label}_sd":   rec.std(axis=0, ddof=1),
    }).set_index("class")

ceB   = [r for r in stab if r["protocol"] == "B"]
tbl = pd.concat([
    perclass_table(ceB,   "CE_final"),
    perclass_table(focal, "Focal"),
], axis=1)

support = {c: np.array(ceB[0]["confusion_matrix"]).sum(axis=1)[i]
           for i, c in enumerate(CLASS_NAMES)}
tbl.insert(0, "support", [support[c] for c in CLASS_NAMES])
print(tbl.to_string(float_format=lambda v: f"{v:.4f}"))

            support  CE_final_mean  CE_final_sd  Focal_mean  Focal_sd
class                                                                
Benign        30000         0.8063       0.1161      0.7631    0.0376
BruteForce      645         0.1499       0.0363      0.5683    0.1332
DDoS          30000         0.8412       0.1602      0.6665    0.0716
DoS           30000         0.8455       0.2364      0.8963    0.0562
Mirai         30000         0.9952       0.0018      0.9942    0.0004
Recon         17834         0.6250       0.1131      0.5599    0.0388
Spoofing      24674         0.7362       0.0821      0.6122    0.0218
Web            1231         0.0345       0.0147      0.6108    0.1281


In [5]:
from scipy import stats

print("Per-class recall variability, CE protocol B (final epoch)\n")
rec = np.array([recalls_from_cm(r["confusion_matrix"]) for r in ceB])
recF = np.array([recalls_from_cm(r["confusion_matrix"]) for r in focal])

for i, c in enumerate(CLASS_NAMES):
    a, b = rec[:, i], recF[:, i]
    u = stats.mannwhitneyu(a, b)
    print(f"{c:12s} CE {a.mean():.4f}+/-{a.std(ddof=1):.4f}   "
          f"Focal {b.mean():.4f}+/-{b.std(ddof=1):.4f}   "
          f"diff {b.mean()-a.mean():+.4f}  p={u.pvalue:.3f}")

Per-class recall variability, CE protocol B (final epoch)

Benign       CE 0.8063+/-0.1161   Focal 0.7631+/-0.0376   diff -0.0432  p=0.005
BruteForce   CE 0.1499+/-0.0363   Focal 0.5683+/-0.1332   diff +0.4184  p=0.000
DDoS         CE 0.8412+/-0.1602   Focal 0.6665+/-0.0716   diff -0.1748  p=0.001
DoS          CE 0.8455+/-0.2364   Focal 0.8963+/-0.0562   diff +0.0508  p=0.525
Mirai        CE 0.9952+/-0.0018   Focal 0.9942+/-0.0004   diff -0.0010  p=0.001
Recon        CE 0.6250+/-0.1131   Focal 0.5599+/-0.0388   diff -0.0652  p=0.003
Spoofing     CE 0.7362+/-0.0821   Focal 0.6122+/-0.0218   diff -0.1240  p=0.000
Web          CE 0.0345+/-0.0147   Focal 0.6108+/-0.1281   diff +0.5764  p=0.000
